In [ ]:
# Per-class analysis of CellWhisperer evaluation results
# Compare trimodal vs bimodal model performance at the class level
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import numpy as np
from collections import defaultdict
import re

print("Starting per-class CellWhisperer analysis...")

In [ ]:
# Load CellWhisperer evaluation results from performance_metrics_permetadata files
results = []

# Parse input files to extract model type, dataset, and metadata
for result_file in snakemake.input:
    df = pd.read_csv(result_file)

    # Extract model type, dataset, and metadata from file path
    path_parts = Path(result_file).parts

    # Find model name (spotwhisperer_*)
    model_name = None
    for part in path_parts:
        if part.startswith("spotwhisperer_") and part != "spotwhisperer_eval":
            model_name = part.replace("spotwhisperer_", "")
            break

    # Extract dataset and metadata column
    dataset_idx = None
    for i, part in enumerate(path_parts):
        if part == "datasets":
            dataset_idx = i + 1
            break

    if dataset_idx and dataset_idx < len(path_parts):
        dataset = path_parts[dataset_idx]
        metadata_col = (
            path_parts[dataset_idx + 1]
            if dataset_idx + 1 < len(path_parts)
            else "celltype"
        )
    else:
        continue

    # Determine model type based on dataset combination
    if model_name == "cellxgene_census__archs4_geo__hest1k__quilt1m":
        model_type = "trimodal"
    elif model_name == "hest1k__quilt1m":
        model_name = "bimodal_bridge"
    elif model_name == "cellxgene_census__archs4_geo":
        model_type = (
            "bimodal_matching"
            if dataset
            in [
                "tabula_sapiens",
                "tabula_sapiens_well_studied_celltypes",
                "pancreas",
                "human_disease",
                "immgen",
            ]
            else "bimodal_mismatch1"
        )
    elif model_name == "hest1k":
        model_type = "bimodal_mismatch1"
    elif model_name == "quilt1m":
        model_type = "bimodal_mismatch2"
    else:
        model_type = "unknown"

    # Add metadata to each row
    df["model_type"] = model_type
    df["model_name"] = model_name
    df["dataset"] = dataset
    df["metadata_col"] = metadata_col
    df["result_file"] = str(result_file)

    results.append(df)

combined_df = pd.concat(results, ignore_index=True)
print(f"Loaded {len(combined_df)} per-class metric entries from {len(results)} files")
print(combined_df.head())

In [ ]:
combined_df[[v for v in combined_df.columns if not v.startswith("n_sam")]]

In [ ]:
combined_df[[v for v in combined_df.columns if not v.startswith("n_sam")]]

In [ ]:
# Create comparison between trimodal and bimodal models for all metrics
combined_df = combined_df.reset_index()
comparison_dfs = []

for dataset in combined_df["dataset"].unique():
    dataset_data = combined_df[combined_df["dataset"] == dataset].copy()

    # Check available metrics
    available_metrics = [
        col for col in dataset_data.columns if col in snakemake.params.metrics
    ]
    print(f"Available metrics for {dataset}: {available_metrics}")

    for metric in available_metrics:
        # Pivot to get model types as columns for each cell type
        pivot_df = dataset_data.pivot_table(
            index=["dataset", "class"],
            columns="model_type",
            values=metric,
            aggfunc="mean",
        ).reset_index()

        # Calculate improvement for this metric
        if "trimodal" in pivot_df.columns and "bimodal_matching" in pivot_df.columns:
            pivot_df[f"{metric}_improvement"] = (
                pivot_df["trimodal"] - pivot_df["bimodal_matching"]
            )
            pivot_df[f"{metric}_relative_improvement"] = (
                pivot_df[f"{metric}_improvement"] / pivot_df["bimodal_matching"] * 100
            )
            pivot_df["metric"] = metric
            comparison_dfs.append(pivot_df)

model_comparison = pd.concat(comparison_dfs, ignore_index=True)
print("Model comparison created for all metrics:")
print(model_comparison.head())

In [ ]:
from matplotlib.backends.backend_pdf import PdfPages

# Get all available metrics from the data
available_metrics = [
    col for col in combined_df.columns if col in snakemake.params.metrics
]
print(f"Available metrics to plot: {available_metrics}")

with PdfPages(snakemake.output.plot) as pdf:
    for dataset, model_comparison_dataset in model_comparison.groupby("dataset"):

        # Plot for each available metric
        for metric in available_metrics:
            metric_data = model_comparison_dataset[
                model_comparison_dataset["metric"] == metric
            ]
            if len(metric_data) == 0:
                continue

            fig, axes = plt.subplots(2, 2, figsize=(20, 12))
            fig.suptitle(
                f"Dataset: {dataset} - {metric.upper()} Performance: Trimodal vs Bimodal Models",
                fontsize=16,
            )

            # Plot 1: Paired barplot comparing bimodal vs trimodal (NEW)
            top_classes = metric_data.nlargest(15, f"{metric}_improvement")
            x = np.arange(len(top_classes))
            width = 0.35

            axes[0, 0].bar(
                x - width / 2,
                top_classes["bimodal_matching"],
                width,
                label="Bimodal",
                alpha=0.8,
                color="skyblue",
            )
            axes[0, 0].bar(
                x + width / 2,
                top_classes["trimodal"],
                width,
                label="Trimodal",
                alpha=0.8,
                color="orange",
            )
            axes[0, 0].set_xlabel("Cell Types")
            axes[0, 0].set_ylabel(f"{metric.upper()} Score")
            axes[0, 0].set_title(
                f"Top 15 Improving Classes: {metric.upper()} Comparison"
            )
            axes[0, 0].set_xticks(x)
            axes[0, 0].set_xticklabels(
                [f"{row['class']}" for _, row in top_classes.iterrows()],
                rotation=45,
                ha="right",
            )
            axes[0, 0].legend()
            axes[0, 0].grid(True, alpha=0.3)

            # Plot 2: Top 10 improving classes (improvement difference)
            top_improving = metric_data.nlargest(10, f"{metric}_improvement")
            y_pos = np.arange(len(top_improving))
            axes[0, 1].barh(
                y_pos, top_improving[f"{metric}_improvement"], color="green", alpha=0.7
            )
            axes[0, 1].set_yticks(y_pos)
            axes[0, 1].set_yticklabels(
                [f"{row['class']}" for _, row in top_improving.iterrows()]
            )
            axes[0, 1].set_xlabel(f"{metric.upper()} Score Improvement")
            axes[0, 1].set_title(
                f"Top 10 Classes: {metric.upper()} Improvement\n(Trimodal - Bimodal)"
            )
            axes[0, 1].grid(True, alpha=0.3)

            # Plot 3: Top 10 declining classes
            top_declining = metric_data.nsmallest(10, f"{metric}_improvement")
            y_pos = np.arange(len(top_declining))
            axes[1, 0].barh(
                y_pos, top_declining[f"{metric}_improvement"], color="red", alpha=0.7
            )
            axes[1, 0].set_yticks(y_pos)
            axes[1, 0].set_yticklabels(
                [f"{row['class']}" for _, row in top_declining.iterrows()]
            )
            axes[1, 0].set_xlabel(f"{metric.upper()} Score Change")
            axes[1, 0].set_title(
                f"Top 10 Classes: {metric.upper()} Decline\n(Trimodal - Bimodal)"
            )
            axes[1, 0].grid(True, alpha=0.3)

            # Plot 4: Distribution of improvements
            axes[1, 1].hist(
                metric_data[f"{metric}_improvement"], bins=20, alpha=0.7, color="blue"
            )
            axes[1, 1].axvline(x=0, color="red", linestyle="--", alpha=0.8)
            axes[1, 1].set_xlabel(f"{metric.upper()} Score Improvement")
            axes[1, 1].set_ylabel("Number of Classes")
            axes[1, 1].set_title(f"Distribution of {metric.upper()} Improvements")
            axes[1, 1].grid(True, alpha=0.3)

            plt.tight_layout()
            pdf.savefig(fig)
            plt.close(fig)

In [ ]:
# Statistical analysis of improvements for all metrics
print("=== CELLWHISPERER STATISTICAL SUMMARY (ALL METRICS) ===")

for metric in snakemake.params.metrics:
    metric_data = model_comparison[model_comparison["metric"] == metric]
    if len(metric_data) == 0:
        continue

    improvements = metric_data[f"{metric}_improvement"].dropna()

    print(f"\n{metric.upper()} Score Improvements:")
    print(f"Mean improvement: {improvements.mean():.4f}")
    print(f"Median improvement: {improvements.median():.4f}")
    print(f"Std deviation: {improvements.std():.4f}")
    print(
        f"Classes with positive improvement: {(improvements > 0).sum()}/{len(improvements)}"
    )
    print(f"Max improvement: {improvements.max():.4f}")
    print(f"Min improvement: {improvements.min():.4f}")

    # Dataset-specific analysis for this metric
    for dataset in metric_data["dataset"].unique():
        dataset_metric_data = metric_data[metric_data["dataset"] == dataset]
        dataset_improvements = dataset_metric_data[f"{metric}_improvement"].dropna()

        if len(dataset_improvements) == 0:
            continue

        positive_improvements = (dataset_improvements > 0).sum()

        print(f"\n{dataset.upper()} Dataset - {metric.upper()}:")
        print(f"  Mean improvement: {dataset_improvements.mean():.4f}")
        print(
            f"  Classes improved: {positive_improvements}/{len(dataset_improvements)}"
        )

        # Top improving classes for this metric
        top_improved = dataset_metric_data.nlargest(3, f"{metric}_improvement")
        print(f"  Top improving classes:")
        for _, row in top_improved.iterrows():
            print(f'    {row["class"]}: +{row[f"{metric}_improvement"]:.4f}')

# Save detailed results
model_comparison.to_csv(snakemake.output.analysis, index=False)
print(f"\nDetailed results saved to: {snakemake.output.analysis}")